In [3]:
import cv2 as cv
import numpy as np

In [4]:
def to_gray(img):
    return cv.cvtColor(img, cv.COLOR_BGR2GRAY)

def print_shapes(*args):
    for a in args:
        print("->", a.shape, "|", a.dtype, end="\n")

## Lucas-Kanade Optical Flow in OpenCV

In [4]:
cap = cv.VideoCapture('../../../class.vision/videos/slow_traffic_small.mp4')

feature_params = dict( maxCorners = 100,
                       qualityLevel = 0.3,
                       minDistance = 7,
                       blockSize = 7 )

lk_params = dict( winSize  = (15, 15),
                  maxLevel = 2,
                  criteria = (cv.TERM_CRITERIA_EPS | cv.TERM_CRITERIA_COUNT, 10, 0.03))

colors = np.random.randint(0, 255, (200, 3), dtype = "uint8")

ret, frame0 = cap.read()
old_gray = to_gray(frame0.copy())
mask = np.zeros_like(frame0)

roi_mask = np.zeros(frame0.shape[:2], 'uint8')
cv.rectangle(roi_mask, (200, 100), (400, 300), 255, -1)

p0 = cv.goodFeaturesToTrack(old_gray, mask=roi_mask, **feature_params).ravel().reshape(-1,2)
print_shapes(p0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_gray = to_gray(frame)
    # print("len p0", len(p0))
    p1, st, err = cv.calcOpticalFlowPyrLK(old_gray, frame_gray, p0, None, **lk_params)

    if p1 is None:
        p1 = p1[st == 1]
        p0 = p0[st == 1]
    for i, (new, old) in enumerate(zip(p1, p0)):
        a, b = new
        c, d = old
        mask = cv.line(mask, (int(a), int(b)), (int(c), int(d)), colors[i].tolist(), 2)
        frame = cv.circle(frame, (int(a), int(b)), 5, colors[i].tolist(), -1)
    img = cv.add(frame, mask)

    cv.imshow('frame', img)
    if cv.waitKey(5) & 0xFF == 27:
        break

    old_gray = frame_gray
    p0 = p1


cap.release()
cv.destroyAllWindows()

-> (9, 2) | float32


## Dense Optical Flow in OpenCV

In [3]:
cap = cv.VideoCapture("../../../class.vision/videos/vtest.avi")
ret, frame0 = cap.read()
prev_frame = to_gray(frame0)

hsv_mask = np.zeros_like(frame0)
hsv_mask[..., 1] = 255

while 1:
    ret, frame = cap.read()
    if not ret:
        break

    next_frame = to_gray(frame)
    flow = cv.calcOpticalFlowFarneback(prev_frame, next_frame, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    if flow is None:
        break

    mag, ang = cv.cartToPolar(flow[..., 0], flow[..., 1])

    hsv_mask[..., 0] = ang * 180 / np.pi / 2
    hsv_mask[..., 2] = cv.normalize(mag, None, 0, 255, cv.NORM_MINMAX)

    bgr = cv.cvtColor(hsv_mask, cv.COLOR_HSV2BGR)
    cv.imshow('frame2', bgr)
    if cv.waitKey(5) & 0xFF == 27:
        break

    prev_frame = next_frame

cap.release()
cv.destroyAllWindows()

## Test with WebCam

In [5]:
def draw_flow(img, flow, step=16):

    h, w = img.shape[:2]
    y, x = np.mgrid[step/2:h:step, step/2:w:step].reshape(2,-1).astype(int)
    fx, fy = flow[y,x].T

    lines = np.vstack([x, y, x-fx, y-fy]).T.reshape(-1, 2, 2)
    lines = np.int32(lines + 0.5)

    img_bgr = cv.cvtColor(img, cv.COLOR_GRAY2BGR)
    cv.polylines(img_bgr, lines, 0, (0, 255, 0))

    for (x1, y1), (_x2, _y2) in lines:
        cv.circle(img_bgr, (x1, y1), 1, (0, 255, 0), -1)

    return img_bgr


def draw_hsv(flow):
    h, w = flow.shape[:2]
    fx, fy = flow[..., 0], flow[..., 1]
    ang = np.arctan2(fy, fx) + np.pi
    mag = np.sqrt(fx ** 2 + fy ** 2)
    hsv = np.zeros((h, w, 3), np.uint8)
    hsv[..., 1] = 255
    hsv[..., 0] = ang * (180 / np.pi / 2)
    hsv[..., 2] = cv.normalize(mag, None, 0, 255, cv.NORM_MINMAX)
    bgr = cv.cvtColor(hsv, cv.COLOR_HSV2BGR)
    return bgr

cap = cv.VideoCapture(0)
ret, frame0 = cap.read()
prev_frame = to_gray(frame0)

while 1:
    ret, frame = cap.read()
    if not ret:
        break

    next_frame = to_gray(frame)
    flow = cv.calcOpticalFlowFarneback(prev_frame, next_frame, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    if flow is None:
        break

    cv.imshow('flow', draw_flow(next_frame, flow))
    hsv_mask = draw_hsv(flow)
    # cv.imshow('hsv', hsv_mask)
    cv.imshow('frame', cv.add(frame,hsv_mask))

    if cv.waitKey(10) & 0xFF == 27:
        break

    prev_frame = next_frame

cap.release()
cv.destroyAllWindows()
